# Notebook Analyse Go/NoGo

Ce Notebook guide l'analyse des potentiels évoqués (ERP) pour le paradigme Go/NoGo.

### Version 2

Il s'agit de la deuxième version de ce notebook. Elle inclut quelques modifications et correctifs pour résoudre des erreurs mineures. Le flux de travail général reste identique à la version précédente.

### Remarque importante

La démarche de ce notebook consiste d'abord à explorer et tester le code **individuellement sur un seul sujet**.

Une fois les étapes validées, une **fonction finale** regroupe tous les blocs de code. Cette fonction permet d'appliquer l'ensemble du pipeline de traitement à **tous les sujets** de manière automatisée.

> **Attention :** Si vous avez expérimenté avec différents paramètres (par exemple, des seuils de filtrage ou des critères de rejet) et que vous souhaitez utiliser des valeurs autres que celles définies par défaut, **ou si vous avez ajouté d'autres blocs de code ou de nouvelles méthodes**, n'oubliez pas de **mettre à jour la fonction finale** en conséquence avant de l'exécuter sur l'ensemble des données.

Le notebook reprend la structure "Type 1 / Type 2 / Type 3" introduite dans `01_preprocessing_notebook.ipynb` :
- **Type 1 — prêt à exécuter** : cellules complètes, sans modification nécessaire.
- **Type 2 — à personnaliser** : blocs contenant des paramètres à ajuster (balises `A_COMPLETER`).
- **Type 3 — exploration libre** : propositions d'analyses supplémentaires, prêtes à modifier.

## Pour bien démarrer
- Assurez-vous d'avoir exécuté le pipeline de prétraitement pour générer les fichiers `*_clean_(eeg ou epo).fif`.
- Activez l'environnement Python du cours et installez les dépendances (`pip install -r requirements.txt`).

## Objectifs pédagogiques
1. Charger un enregistrement Go/NoGo déjà prétraité.
2. Extraire les événements, découper en epochs et calculer les ERPs Go vs NoGo.
3. Explorer des métriques temporelles (amplitude moyenne, latence de pics) et comparer plusieurs sujets ou données de groupe.


## 0. Préparation et configuration

Nous commençons par configurer l'environnement d'analyse (imports, chemins, sélection du sujet).


### Bloc Type 1 — Imports et options globales

Initialise les bibliothèques nécessaires, masque certains avertissements MNE et fixe le style de figures Matplotlib.


In [ ]:
# -----------------------------------------------------------------------------
# Imports principaux et configuration globale
# Chaque instruction est commentée pour rappeler son rôle.
# -----------------------------------------------------------------------------
import warnings  # contrôle de l'affichage des avertissements Python
from pathlib import Path  # manipulation portable des chemins
import csv  # lecture de fichiers tabulés (.tsv)
import json  # sauvegarde des résultats intermédiaires

import matplotlib.pyplot as plt  # production des figures
import mne  # bibliothèque cœur pour les analyses EEG/MEG
import numpy as np  # opérations numériques vectorisées
import mne_bids  # outils pour gérer la structure BIDS avec MNE
from mne_bids import BIDSPath  # construction de chemins compatibles BIDS

warnings.filterwarnings('ignore', category=RuntimeWarning)  # masque certains avertissements MNE
mne.set_log_level('INFO')  # verbosité modérée pour suivre les étapes clés
plt.rcParams['figure.figsize'] = (10, 5)  # taille par défaut des figures Matplotlib

print('Versions utilisées:')  # journalise les versions pour la reproductibilité
print(' - mne      ', mne.__version__)  # version de MNE
print(' - numpy    ', np.__version__)  # version de NumPy
print(' - mne_bids ', mne_bids.__version__)  # version de mne-bids


### Bloc Type 1 — Définir le dossier BIDS et les dérivés

Localise le dossier BIDS (données brutes) ainsi que les dérivés produits par le pipeline :
- `derivatives/preproc` pour les fichiers prétraités (sortie du notebook 01),
- `derivatives/gonogo-erp` pour les ERPs de groupe fournis,
- `derivatives/gonogo-analysis` pour les résultats sauvegardés par ce notebook.


In [ ]:
# -----------------------------------------------------------------------------
# Localisation des données BIDS et des dérivés nécessaires
# -----------------------------------------------------------------------------
root_bids = Path('tasks/gonogo/bids')  # chemin relatif recommandé

print('Chemins vérifiés:')  # confirmation dans la console
print(' - BIDS root            :', root_bids.resolve())  # chemin absolu utilisé

deriv_preproc = root_bids / 'derivatives' / 'preproc'  # mêmes dérivés que le notebook de prétraitement
deriv_preproc.mkdir(parents=True, exist_ok=True)

deriv_erp = root_bids / 'derivatives' / 'gonogo-erp'  # ERP de groupe fournis

deriv_analysis = root_bids / 'derivatives' / 'gonogo-analysis'  # résultats spécifiques à ce notebook
deriv_analysis.mkdir(parents=True, exist_ok=True)
print(' - Dérivés (preproc)    :', deriv_preproc.resolve())
print(' - Dérivés (analysis)   :', deriv_analysis.resolve())


### Bloc Type 1 — Lister les participants disponibles

Lit `participants.tsv` (convention BIDS) pour récupérer les identifiants `sub-XX` disponibles. 
En Type 2 ci-dessous, nous choisirons l'un de ces sujets.


In [ ]:
# -----------------------------------------------------------------------------
# Lecture de participants.tsv afin d'obtenir la liste des sujets présents
# -----------------------------------------------------------------------------
participants_tsv = root_bids / 'participants.tsv'  # chemin vers le fichier BIDS
subjects = []  # contiendra les identifiants sans le préfixe 'sub-'

with participants_tsv.open('r', encoding='utf-8') as f:  # ouverture du fichier en lecture
    reader = csv.reader(f, delimiter='	')  # lecture tabulée
    header = next(reader, None)  # saute l'entête (participant_id, ...)
    for row in reader:  # boucle sur chaque ligne restante
        if not row:  # ignore les lignes vides
            continue
        participant_id = row[0]  # première colonne = identifiant sujet
        if participant_id.startswith('sub-'):  # vérifie le format BIDS
            subjects.append(participant_id.replace('sub-', ''))  # stocke l'identifiant sans préfixe

print(f"Participants détectés ({len(subjects)}): {subjects}")  # affiche la liste obtenue


### Bloc Type 2 — Sélectionner un participant et une session

Modifiez les variables ci-dessous pour cibler un autre enregistrement. 
Les valeurs proposées correspondent à un exemple valide, mais n'hésitez pas à tester différents sujets.


In [ ]:
# -----------------------------------------------------------------------------
# Choix du sujet/session/run à analyser (modifiable)
# -----------------------------------------------------------------------------
# On va commenecer par un sujet unique pour simplifier l'exemple. A la fin du notebook,
# on a une fonction qui permet de boucler sur tous les sujets.
subject = '03'  # <--- remplacez par ex. '05' pour explorer un autre participant
session = '001'  # on n'a qu'une session dans ce dataset
run = '01'  # un seul run Go/NoGo disponible

print(f'Sujet en cours: sub-{subject}, session {session}, run {run}')  # trace la sélection


## 1. Charger un enregistrement prétraité

Nous chargeons le fichier `*_clean.fif` issu du pipeline précédent, appliquons un montage standard et vérifions les métadonnées clés.


### Bloc Type 1 — Fonction utilitaire de chargement

`load_processed_raw` centralise la construction du chemin de fichier, le chargement `Raw` MNE et l'application d'un montage 10-20 international.


In [ ]:
# -----------------------------------------------------------------------------
# Fonction utilitaire : chargement d'un fichier prétraité pour un sujet donné
# -----------------------------------------------------------------------------
def load_processed_raw(subject: str, session: str = '001', run: str = '01') -> mne.io.BaseRaw:
    # Construit le chemin BIDS du fichier *_processed.fif dans derivatives/preproc
    processed_bids = BIDSPath(
        root=deriv_preproc,
        subject=subject,
        session=session,
        task='gonogo',
        run=run,
        datatype='eeg',
        suffix='eeg',
        processing='clean',
        extension='.fif'
    )
    fname = processed_bids.fpath
    if not fname.exists():  # garde-fou si le fichier manque
        raise FileNotFoundError(f'Fichier introuvable: {fname}')  # message explicite
    raw_obj = mne.io.read_raw_fif(fname, preload=True)  # charge en mémoire pour un accès rapide
    raw_obj.set_montage('standard_1020', match_case=False, on_missing='warn')  # assure la co-registration EEG
    return raw_obj  # renvoie l'objet Raw prêt à l'emploi

raw = load_processed_raw(subject, session=session, run=run)  # chargement effectif
print(raw)  # résumé de l'objet Raw


### Bloc Type 1 — Vérifier les annotations et métadonnées

Affiche la fréquence d'échantillonnage, la liste des canaux EEG, ainsi que les annotations importées (événements détectés et autres marquages).


In [ ]:
raw.plot(start=98, duration=5)  # décommentez pour inspecter visuellement

In [ ]:
# -----------------------------------------------------------------------------
# Inspection rapide des métadonnées pour valider le chargement
# -----------------------------------------------------------------------------
print('Fréquence échantillonnage :', raw.info['sfreq'], 'Hz')  # vérifie la fréquence
print('Annotations disponibles   :', sorted(set(raw.annotations.description)))  # types d'événements
print('Nombre total annotations  :', len(raw.annotations))  # quantité d'annotations


### Bloc Type 2 — Visualisation rapide du signal brut (optionnel)

Décommentez la ligne suivante pour afficher quelques secondes de signal. 
Ajustez `n_channels`, `scalings` ou la fenêtre temporelle selon vos besoins.


In [ ]:
# raw.copy().crop(tmax=5).plot(n_channels=12, scalings='auto')

## 2. Détection des événements et création des epochs

Nous mappons les annotations BIDS vers des étiquettes lisibles (`go/onset`, `nogo/onset`, ...), puis nous découpons les données en epochs alignés sur ces événements.


### Bloc Type 1 — Extraire les événements Go/NoGo depuis les annotations

La table `annotation_map` relie les identifiants bruts (`Stimulus/S  4`, `Stimulus/S  5`, ...) à des labels plus explicites.


In [ ]:
import pandas as pd
# -----------------------------------------------------------------------------
# Étape 1 : Extraction initiale et définition des codes
# -----------------------------------------------------------------------------
events, event_id = mne.events_from_annotations(raw)

# Nouveaux codes pour nos événements
new_event_codes = {
    'stimulus/go/correct': 101,
    'stimulus/go/incorrect': 102, # Erreur d'omission (pas de réponse)
    'stimulus/nogo/correct': 103, # Omission correcte
    'stimulus/nogo/incorrect': 104 # Erreur de commission (réponse erronée)
}

# Dictionnaire pour MNE, qui sera rempli dynamiquement
selected_event_id = {}

# Obtenir les anciens codes sources (en utilisant .get())
go_onset_code = event_id.get(np.str_('go_onset'))
nogo_onset_code = event_id.get(np.str_('nogo_onset'))
go_correct_code = event_id.get(np.str_('go_correct'))
nogo_correct_code = event_id.get(np.str_('nogo_correct'))
incorrect_code = event_id.get(np.str_('incorrect'))

# Codes de stimulus et de réponse
stimulus_codes = [go_onset_code, nogo_onset_code]
response_codes = [go_correct_code, nogo_correct_code, incorrect_code]

# -----------------------------------------------------------------------------
# Étape 2 : Boucle pour renommer les événements
# -----------------------------------------------------------------------------
print("Analyse de la séquence d'événements pour créer des essais stimulus-locked...")

new_events_list = [] # Notre future matrice d'événements
behavioral_data = [] # Pour stocker les TR et la justesse
sfreq = raw.info['sfreq']
trial_count = 0

for i in range(len(events)):
    # On ne s'intéresse qu'au DÉBUT d'un essai (un stimulus onset)
    current_code = events[i, 2]
    if current_code not in stimulus_codes:
        continue
        
    # --- On a trouvé un stimulus ! ---
    trial_count += 1
    t_onset = events[i, 0] # L'échantillon (sample) du stimulus
    stim_type = 'go' if current_code == go_onset_code else 'nogo'
    
    # --- Maintenant, "regardons en avant" pour trouver la réponse ---
    found_response = False
    for j in range(i + 1, len(events)):
        next_code = events[j, 2]
        
        # Cas 1 : On trouve la réponse
        if next_code in response_codes:
            found_response = True
            t_response = events[j, 0]
            rt_sec = (t_response - t_onset) / sfreq
            
            if stim_type == 'go':
                if next_code == go_correct_code:
                    # Essai Go Réussi
                    new_code = new_event_codes['stimulus/go/correct']
                    response = 'correct'
                else:
                    # Essai Go Échoué (ex: 'incorrect')
                    new_code = new_event_codes['stimulus/go/incorrect']
                    response = 'incorrect'
            
            else: # stim_type == 'nogo'
                if next_code == nogo_correct_code:
                    # Essai NoGo Réussi (l'annotation 'nogo_correct' existe)
                    new_code = new_event_codes['stimulus/nogo/correct']
                    response = 'correct'
                    rt_sec = np.nan # Pas de TR pour une non-réponse
                else:
                    # Essai NoGo Échoué (ex: 'incorrect' ou 'go_correct')
                    # C'est une ERREUR DE COMMISSION
                    new_code = new_event_codes['stimulus/nogo/incorrect']
                    response = 'incorrect'
            
            # On a notre réponse, on sort de la boucle "look-ahead"
            break
            
        # Cas 2 : On trouve le PROCHAIN stimulus (essai manqué)
        if next_code in stimulus_codes:
            # On a atteint le stimulus suivant sans trouver de réponse
            found_response = False
            break

    # Cas 3 : On n'a pas trouvé de réponse avant la fin (ou avant le prochain stimulus)
    if not found_response:
        rt_sec = np.nan # Pas de réponse
        
        if stim_type == 'go':
            # Essai Go manqué -> ERREUR D'OMISSION
            new_code = new_event_codes['stimulus/go/incorrect']
            response = 'incorrect'
        else: # stim_type == 'nogo'
            # Essai NoGo manqué (pas de réponse) -> C'EST CORRECT
            new_code = new_event_codes['stimulus/nogo/correct']
            response = 'correct'

    # --- Ajouter l'événement STIMULUS-LOCKED renommé ---
    # On ajoute le [t_onset, 0, new_code]
    new_events_list.append([t_onset, 0, new_code])
    
    # --- Ajouter les données comportementales ---
    behavioral_data.append({
        'trial': trial_count,
        'stimulus': stim_type,
        'response': response,
        'rt_sec': rt_sec
    })

# -----------------------------------------------------------------------------
# Étape 3 : Finalisation et Résumé
# -----------------------------------------------------------------------------

# Convertir la liste en matrice numpy, comme MNE s'y attend
events_modified = np.array(new_events_list)

# Créer le dictionnaire event_id final
selected_event_id = {label: code for label, code in new_event_codes.items() 
                     if code in events_modified[:, 2]} # Ne garde que les événements existants

# Convertir les données comportementales en DataFrame Pandas pour inspection
df_behavior = pd.DataFrame(behavioral_data)

print("\n--- Résumé du Renommage Stimulus-Locked ---")
print(f"Total d'essais (stimulus) trouvés : {trial_count}")
print("Nouveaux événements créés :")
for label, code in selected_event_id.items():
    n_trials = (events_modified[:, 2] == code).sum()
    print(f" - {label:25s} → code {code}, essais = {n_trials}")

print("\n--- Données Comportementales (Aperçu) ---")
print(df_behavior.head())

print("\nTemps de réaction moyen (Go correct) :")
mean_rt = df_behavior[
    (df_behavior['stimulus'] == 'go') & (df_behavior['response'] == 'correct')
]['rt_sec'].mean()
print(f"{mean_rt * 1000:.2f} ms")

### Bloc Type 1 — Paramètres d'epoching

Définit la fenêtre temporelle autour des événements et calcule les epochs EEG associés.


In [ ]:
# -----------------------------------------------------------------------------
# Paramètres d'epoching et construction des epochs MNE
# -----------------------------------------------------------------------------

# Définition de la fenêtre temporelle pour chaque epoch :
tmin, tmax = -0.2, 0.8  # -200 ms avant l'événement (t=0) et +800 ms après.

# Définition de la période de "ligne de base" (baseline) :
baseline = (-0.2, 0.0)  # Utilise l'intervalle [-200 ms, 0 ms] (pré-stimulus)
                        # La moyenne de cette période sera soustraite de toute l'epoch.

# Création de l'objet Epochs
epochs = mne.Epochs(
    raw,  # Le signal continu (filtré) à découper
    events_modified,  # UTILISATION CORRECTE : La matrice des événements modifiée
    event_id=selected_event_id,  # UTILISATION CORRECTE : Le dictionnaire propre
    tmin=tmin,  # Début de la fenêtre
    tmax=tmax,  # Fin de la fenêtre
    baseline=baseline,  # Période de correction de la ligne de base
    picks='eeg',  # Sélectionne UNIQUEMENT les canaux de type 'eeg'
    preload=True,  # Charge toutes les données des epochs en mémoire (RAM).
                  # REQUIS pour AutoReject, ICA, etc.
    detrend=None,  # N'applique pas de "detrending"
)

print(epochs)  # Affiche un résumé (nombre d'epochs, canaux, temps)

# Comptage final des essais pour chaque condition
# (en utilisant les étiquettes de 'selected_event_id')
trial_counts = {cond: len(epochs[cond]) for cond in epochs.event_id}

print('Essais conservés par condition :', trial_counts)  # Affichage console

### Bloc Type 2 — Visualiser quelques epochs (optionnel)

Décommentez le code ci-dessous pour afficher un extrait d'epochs et inspecter la qualité des essais.


In [ ]:
# epochs.copy().plot(n_epochs=10, n_channels=10, scalings='auto')  # décommentez pour inspecter visuellement

### Bloc Type 1 — Calculer les ERPs Go vs NoGo

Calcule la moyenne des epochs pour chaque condition et produit un tracé temporel.


In [ ]:
# -----------------------------------------------------------------------------
# --- Création et Visualisation des Potentiels Évoqués (ERPs) ---
# -----------------------------------------------------------------------------

# --- Calcul des ERPs de base (Inspiré de votre exemple) ---
#
# Utilise une "dictionary comprehension" pour calculer la moyenne (Evoked)
# de chaque condition définie dans 'selected_event_id'.
evokeds = {label: epochs[label].average() for label in epochs.event_id.keys()}

print(f"Evokeds de base calculés : {list(evokeds.keys())}")

# --- Visualisation individuelle de chaque ERP (Inspiré de votre exemple) ---
#
# Boucle sur chaque ERP et affiche son tracé "papillon" (butterfly plot),
# montrant tous les canaux.
print("\n--- Visualisation individuelle de chaque ERP ---")
for label, evk in evokeds.items():
    print(f"Affichage de l'ERP pour : {label} (N={len(epochs[label])})")
    # tracé interactif, spatial_colors=True colore chaque canal différemment
    # (Peut générer beaucoup de fenêtres !)
    evk.plot(spatial_colors=True, time_unit='s', titles=f'ERP — {label}', show=True)

### Bloc Type 3 — Visualisations Comparatives (optionnel)

Utilisez et adaptez le code en bas pour faire des comparaisons entre conditions spécifiques, par exemple entre "go correct" et "nogo incorrect".

In [ ]:
# Ici, nous superposons des conditions spécifiques pour voir les différences.
print("\n--- Étape 2c : Génération des graphiques ERP comparatifs ---")

# 1. Comparaison : All Go vs. All NoGo (La principale)
# On doit d'abord créer ces Evokeds de groupe
evk_all_go = epochs['stimulus/go'].average() if 'stimulus/go' in epochs else None
evk_all_nogo = epochs['stimulus/nogo'].average() if 'stimulus/nogo' in epochs else None

if evk_all_go and evk_all_nogo:
    print("Affichage : All Go (bleu) vs. All NoGo (orange)")
    mne.viz.plot_compare_evokeds(
        [evk_all_go, evk_all_nogo],
        title="Comparaison Principale : All Go vs. All NoGo",
        legend='upper left',
        picks='Cz', # Trace seulement 'Cz'.
        show=True
    )
    # 

# 2. Comparaison : Go Correct vs. Go Incorrect
# On utilise les evokeds de base calculés en 2a
if 'stimulus/go/correct' in evokeds and 'stimulus/go/incorrect' in evokeds:
    print("Affichage : Go Correct (bleu) vs. Go Incorrect (orange)")
    mne.viz.plot_compare_evokeds(
        [evokeds['stimulus/go/correct'], evokeds['stimulus/go/incorrect']],
        title="Comparaison : Go Correct vs. Go Incorrect",
        legend='upper left',
        picks='Cz',
        show=True
    )

# 3. Comparaison : NoGo Correct vs. NoGo Incorrect
if 'stimulus/nogo/correct' in evokeds and 'stimulus/nogo/incorrect' in evokeds:
    print("Affichage : NoGo Correct (bleu) vs. NoGo Incorrect (orange)")
    mne.viz.plot_compare_evokeds(
        [evokeds['stimulus/nogo/correct'], evokeds['stimulus/nogo/incorrect']],
        title="Comparaison : NoGo Correct vs. NoGo Incorrect (Erreur)",
        legend='upper left',
        picks='FCz', # L'ERN est souvent maximal sur 'FCz'
        show=True
    )
    #

### Bloc Type 2 — Topographies temporelles


In [ ]:
# -----------------------------------------------------------------------------
# Topographies (Instantanés) sur une fenêtre d'intérêt (200-450 ms)
# -----------------------------------------------------------------------------
#
# Cette méthode ne fait PAS la moyenne de la fenêtre.
# Elle prend 5 "instantanés" (snapshots) équidistants
# pour montrer l'ÉVOLUTION de la topographie durant la fenêtre.

# Définition des 5 instants (en secondes)
times_snapshots = np.linspace(0.2, 0.45, 5)  # 0.2, 0.26, 0.32, 0.38, 0.45 s

# --- Instantanés pour 'Go Correct' ---
if 'stimulus/go/correct' in evokeds:
    print("Affichage des 5 instantanés pour 'Go Correct'")
    evokeds['stimulus/go/correct'].plot_topomap(
        times=times_snapshots, # Les 5 instants
        ch_type='eeg',
        time_unit='s',
        colorbar=True,
    )

# --- Instantanés pour 'NoGo Correct' ---
if 'stimulus/nogo/correct' in evokeds:
    print("Affichage des 5 instantanés pour 'NoGo Correct'")
    evokeds['stimulus/nogo/correct'].plot_topomap(
        times=times_snapshots, # Mêmes instants pour comparer
        ch_type='eeg',
        time_unit='s',
        colorbar=True,
    )

In [ ]:
# -----------------------------------------------------------------------------
# Visualisation A : Topographies MOYENNÉES sur 200-450 ms
# -----------------------------------------------------------------------------
#
# Ici, nous moyennons l'activité sur la fenêtre d'intérêt
# pour obtenir une seule carte par condition.

# Définir le centre (0.325s) et la largeur (0.25s) de la fenêtre
t_center = 0.325 # (0.2 + 0.45) / 2
t_width = 0.25   # 0.45 - 0.2

# --- Moyenne pour 'Go Correct' ---
if 'stimulus/go/correct' in evokeds:
    print("Affichage de la topographie MOYENNE pour 'Go Correct'")
    evokeds['stimulus/go/correct'].plot_topomap(
        times=t_center,     # Le centre de la fenêtre
        average=t_width,    # La largeur de la fenêtre à moyenner
        ch_type='eeg',
        time_unit='s',
        colorbar=True,
    )

# --- Moyenne pour 'NoGo Correct' ---
if 'stimulus/nogo/correct' in evokeds:
    print("Affichage de la topographie MOYENNE pour 'NoGo Correct'")
    evokeds['stimulus/nogo/correct'].plot_topomap(
        times=t_center,
        average=t_width,
        ch_type='eeg',
        time_unit='s',
        colorbar=True,
    )

### Bloc Type 3 — Visualisations Comparatives, Topographie de la DIFFÉRENCE (optionnel)

Utilisez et adaptez le code en bas pour faire des comparaisons entre conditions spécifiques, par exemple entre "go correct" et "nogo incorrect".

In [ ]:
# -----------------------------------------------------------------------------
# Visualisation B : Topographie de la DIFFÉRENCE (NoGo - Go)
# -----------------------------------------------------------------------------
#
# La meilleure façon de comparer est de soustraire un ERP de l'autre.
# Cela crée une "onde de différence" (Difference Wave).
# Nous pouvons ensuite tracer la topographie de cette différence.

if 'stimulus/go/correct' in evokeds and 'stimulus/nogo/correct' in evokeds:
    print("Calcul et affichage de la topographie de la DIFFÉRENCE")
    
    # 1. Créer l'Evoked de différence : (NoGo Correct) - (Go Correct)
    evk_diff = mne.combine_evoked(
        [evokeds['stimulus/nogo/correct'], evokeds['stimulus/go/correct']],
        weights=[1, -1]  # Poids : +1 pour NoGo, -1 pour Go
    )
    
    evk_diff.comment = 'Difference (NoGo - Go)' # Renommer pour la clarté
    
    # 2. Afficher la topographie MOYENNE de cette différence
    #    (C'est souvent le graphique le plus publié)
    evk_diff.plot_topomap(
        times=t_center, # Centre de la fenêtre (0.325 s)
        average=t_width, # Largeur de la fenêtre (0.25 s)
        ch_type='eeg',
        time_unit='s',
        colorbar=True,
        # 'vlim' est important ici pour centrer la couleur sur 0
        vlim=max(np.abs(evk_diff.data.min()), np.abs(evk_diff.data.max())),
    )
    # 
    
    # 3. (Optionnel) Afficher les instantanés de la différence
    evk_diff.plot_topomap(
        times=times_snapshots, # Les 5 instants
        ch_type='eeg',
        time_unit='s',
        colorbar=True,
        vlim=max(np.abs(evk_diff.data.min()), np.abs(evk_diff.data.max())),
    )

## 3. Comparer amplitudes et latences (temps) des ERP


### Fonction d'aide — Fonctions d'extraction de métriques temporelles

- `mean_amplitude_microvolt` calcule la moyenne d'amplitude (µV) sur une fenêtre temporelle.
- `peak_latency` renvoie la latence et l'amplitude du pic positif (ou négatif) demandé.


In [ ]:
# -----------------------------------------------------------------------------
# Définition des Régions d'Intérêt (ROI) Spatiales
# -----------------------------------------------------------------------------
# Ce dictionnaire regroupe des canaux EEG en "clusters" logiques.
# Cela permet d'analyser une région (ex: "parietal") plutôt qu'un seul
# canal, ce qui augmente le rapport signal/bruit (SNR).
CHANNEL_CLUSTERS = {
    # Cluster fronto-central, crucial pour les composantes N2 et ERN
    "midline_fc": ["Fz", "Cz"],
    # Cluster pariétal, crucial pour la composante P3
    "parietal": ["P3", "P4"],
    # Clusters moteurs, pour analyser la préparation et l'exécution motrice
    "motor_left": ["C3"],
    "motor_right": ["C4"],
}

# -----------------------------------------------------------------------------
# Définition des Caractéristiques (Features) basées sur la MOYENNE
# -----------------------------------------------------------------------------
# C'est une "liste de tâches" pour l'extraction d'amplitude MOYENNE.
# Chaque dictionnaire définit une caractéristique à extraire.
ERP_MEAN_WINDOWS = [
    {
        "name": "N2_mean_200_350",         # Nom de la future colonne (feature)
        "cluster": "midline_fc",           # Quel cluster de canaux utiliser
        "tmin": 0.200, "tmax": 0.350       # Fenêtre temporelle (en secondes)
    },
    {
        "name": "P3_mean_300_600_midline_fc",
        "cluster": "midline_fc",
        "tmin": 0.300, "tmax": 0.600
    },
    {
        "name": "P3_mean_300_600_parietal",
        "cluster": "parietal",
        "tmin": 0.300, "tmax": 0.600
    },
]

# -----------------------------------------------------------------------------
# Définition des Caractéristiques (Features) basées sur le PIC
# -----------------------------------------------------------------------------
# Liste de tâches similaire, mais pour l'extraction de PIC (amplitude ET latence).
ERP_PEAK_WINDOWS = [
    {
        "name": "N2_peak_200_350",
        "cluster": "midline_fc",
        "tmin": 0.200, "tmax": 0.350,
        "mode": "min"  # 'min' : cherche le pic négatif (ex: N2)
    },
    {
        "name": "P3_peak_300_600",
        "cluster": "parietal",
        "tmin": 0.300, "tmax": 0.600,
        "mode": "max"  # 'max' : cherche le pic positif (ex: P3)
    },
]

# -----------------------------------------------------------------------------
# Fonction Utilitaire (Helper Function)
# -----------------------------------------------------------------------------
def _window_mask(times: np.ndarray, tmin: float, tmax: float) -> np.ndarray:
    """
    Fonction utilitaire (helper) très rapide.
    
    Prend un array de temps (ex: [-0.2, -0.18, ..., 0.8]) et retourne
    un masque booléen (ex: [False, ..., True, True, ..., False])
    pour tous les points de temps situés entre tmin et tmax.
    
    Permet de sélectionner (slicer) les données EEG très efficacement.
    """
    return (times >= tmin) & (times <= tmax)

In [ ]:
def peak_latency(evoked: mne.Evoked, 
                picks: list | str, 
                tmin: float, 
                tmax: float, 
                mode: str = 'max') -> tuple[float, float]:
    """
    Calcule le pic (amplitude et latence) sur un CLUSTER de canaux.

    Cette fonction moyenne d'abord les canaux du cluster en un seul
    "canal virtuel", PUIS trouve le pic (min ou max) sur ce signal moyen.
    
    Args:
        evoked: Les données EEG (objet MNE Evoked ou Epochs).
        picks: Liste de noms de canaux (ex: ['Pz', 'P3', 'P4']).
        tmin: Début de la fenêtre temporelle (en secondes).
        tmax: Fin de la fenêtre temporelle (en secondes).
        mode: 'max' (pic positif, ex: P3) ou 'min' (pic négatif, ex: N2).

    Returns:
        Un tuple (latence_en_secondes, amplitude_en_µV).
        Ex: (0.345, 4.51)
    """
    # 1. Copie et sélection des canaux et de la fenêtre de temps
    evk = evoked.copy().pick(picks)
    start, stop = evk.time_as_index([tmin, tmax])
            
    # 2. Slicer les données (data shape est [n_canaux, n_temps])
    segment = evk.data[:, start:stop]
    
    # 3. Calculer le "canal virtuel" en moyennant sur l'axe des canaux (axis=0)
    # On obtient un array 1D (le signal moyen du cluster)
    cluster_signal = segment.mean(axis=0)
    
    # 4. Trouver le pic (min ou max) sur ce signal 1D
    if mode == 'max':
        peak_idx = np.argmax(cluster_signal)
        peak_amp_V = np.max(cluster_signal)
    elif mode == 'min':
        peak_idx = np.argmin(cluster_signal)
        peak_amp_V = np.min(cluster_signal)
    else:
        raise ValueError("Mode doit être 'min' ou 'max'")

    # 5. Convertir l'index du pic en temps (secondes)
    # (L'index est relatif à 'start', donc on l'ajoute)
    peak_time_sec = evk.times[start + peak_idx]
    
    return float(peak_time_sec), float(peak_amp_V * 1e6)

def mean_amplitude_microvolt(evoked: mne.Evoked, 
                             picks: list | str, 
                             tmin: float, 
                             tmax: float) -> dict:
    """
    Calcule l'amplitude moyenne pour chaque canal d'un cluster ainsi que
    que au sein du cluster.

    Args:
        evoked: les données EEG (objet MNE Evoked ou Epochs).
        picks: Liste de noms de canaux (ex: ['Cz', 'Pz']) ou un seul (ex: 'Cz').
        tmin: Début de la fenêtre temporelle (en secondes).
        tmax: Fin de la fenêtre temporelle (en secondes).

    Returns:
        Un dictionnaire {canal: amplitude_moyenne_en_µV}.
        Ex: {'Fz': -1.23, 'Cz': -0.98}
    """
    # 1. Copie et sélection des canaux et de la fenêtre de temps
    evk = evoked.copy().pick(picks)
    start, stop = evk.time_as_index([tmin, tmax])

    # 3. Slicer les données (data shape est [n_canaux, n_temps])
    segment = evk.data[:, start:stop]
    
    # 4. Calculer la moyenne sur l'axe du temps (axis=1) pour chaque canal
    # et retourner un dictionnaire converti en microvolts (µV)
    return {
        ch: float(segment[i].mean() * 1e6)  # moyenne µV par canal
        for i, ch in enumerate(evk.ch_names)
    }

### Bloc Type 1 — extraction de métriques temporelles pour stimulus/nogo/correct sur les données 'evoked' des essaie correct de nogo (stimulus/nogo/correct)


In [ ]:
print("--- ANALYSE DES AMPLITUDES MOYENNES (Exploratoire) ---")
for config in ERP_MEAN_WINDOWS:
    cluster_name = config['cluster']
    picks = CHANNEL_CLUSTERS[cluster_name]
    
    # Appel de la fonction 1
    mean_amps = mean_amplitude_microvolt(
        evokeds['stimulus/nogo/correct'], 
        picks=picks, 
        tmin=config['tmin'], 
        tmax=config['tmax']
    )
    
    print(f"Caractéristique: {config['name']}")
    # Calcule la moyenne du dictionnaire pour avoir la moyenne du cluster
    avg_cluster_amp = np.mean(list(mean_amps.values()))
    print(f"  Moyenne du cluster '{cluster_name}': {avg_cluster_amp:.2f} µV")
    print(f"  Détail canaux : {mean_amps}")


print("\n--- ANALYSE DES PICS (Exploratoire) ---")
for config in ERP_PEAK_WINDOWS:
    cluster_name = config['cluster']
    picks = CHANNEL_CLUSTERS[cluster_name]
    
    # Appel de la fonction 2
    latency, amplitude = peak_latency(
        evokeds['stimulus/nogo/correct'],
        picks=picks,
        tmin=config['tmin'],
        tmax=config['tmax'],
        mode=config['mode']
    )
    
    print(f"Caractéristique: {config['name']}")
    print(f"  Pic du cluster '{cluster_name}': {amplitude:.2f} µV @ {latency * 1000:.0f} ms")

### Bloc Type 2 — Appliquer l'extraction de métriques temporelles sur les essais 'Go' corrects (stimulus/go/correct)

In [ ]:
# utilisez le meme code

### Bloc Type 3 — Explorer d'autres options

Pistes d'exploration :

* **Jouer avec les configurations :** Modifier les dictionnaires `CHANNEL_CLUSTERS`, `ERP_MEAN_WINDOWS`, et `ERP_PEAK_WINDOWS`.
* **Analyse par capteur :** Essayer d'analyser chaque capteur individuellement (sans utiliser de clusters).
* **Ajouter des composantes :** Identifier d'autres ERPs (ex: N1, P2) identifiés dans la littérature et les ajouter aux configurations.

#### Note: Inclure d'autre attributs, peut être une piste d'analyse additionelle comme cela permet de comparer les attribute de bases ainsi que d'autres.

## 4. Calculer amplitudes et latences (temps) des epochs pour les utiliser comme attributs (features)

Maintenant que nous avons exploré visuellement nos données en moyennant les essais (avec les objets `Evoked`), nous passons à l'étape cruciale pour le **Machine Learning**.

L'objectif ici n'est plus de regarder la moyenne, mais de **quantifier** l'activité cérébrale pour **chaque essai individuellement**.

Un attribut est une mesure numérique qui résume une information clé sur un essai. Notre hypothèse est que des attributs bien choisis (ex: "l'amplitude de la P3") peuvent aider un modèle à distinguer un essai "Go" d'un essai "NoGo".

Nous allons utiliser les mêmes dictionnaires de configuration (`CHANNEL_CLUSTERS`, `ERP_MEAN_WINDOWS`, `ERP_PEAK_WINDOWS`) et les fonctions d'aides (`mean_amplitude_epochs`, `peak_latency_epochs`) que nous avons définis.

**La Sortie :** Le résultat final sera un **DataFrame `pandas`**.
    * Chaque **ligne** représentera un **essai (epoch)**.
    * Chaque **colonne** représentera un **attribut** (ex: `P3_peak_amplitude_uV` ou `N2_mean_200_350`).

Ce DataFrame sera la matrice `X` (features) et `y` (labels) que nous utiliserons pour entraîner nos modèles de Machine Learning.

In [ ]:
def mean_amplitude_epochs(epochs: mne.Epochs, 
                          picks: list | str, 
                          tmin: float, 
                          tmax: float) -> np.ndarray:
    """
    Extrait l'amplitude moyenne d'un cluster sur une fenêtre de temps 
    pour CHAQUE ESSAI.

    Args:
        epochs: L'objet Epochs (données 3D : n_essais, n_canaux, n_temps).
        picks: Liste des canaux du cluster (ex: ['Pz', 'P3', 'P4']).
        tmin: Début de la fenêtre (secondes).
        tmax: Fin de la fenêtre (secondes).

    Returns:
        Un array 1D (shape [n_essais,]) contenant la moyenne en µV 
        de chaque essai.
    """
    # 1. Trouver les indices de temps
    start, stop = epochs.time_as_index([tmin, tmax])
    
    # 2. Obtenir les données 3D (n_essais, n_canaux_cluster, n_temps_fenetre)
    data = epochs.get_data(picks=picks)[:, :, start:stop]
    
    # 3. Calculer la moyenne sur les canaux (axis=1) ET le temps (axis=2)
    #    On obtient un array 1D (shape [n_essais,])
    mean_per_trial_V = data.mean(axis=(1, 2))
    
    # 4. Convertir en microvolts et retourner
    return mean_per_trial_V * 1e6

def peak_latency_epochs(epochs: mne.Epochs, 
                        picks: list | str, 
                        tmin: float, 
                        tmax: float, 
                        mode: str = 'max') -> tuple[np.ndarray, np.ndarray]:
    """
    Extrait le pic (amplitude et latence) d'un cluster sur une fenêtre 
    de temps pour CHAQUE ESSAI.

    Args:
        epochs: L'objet Epochs (données 3D).
        picks: Liste des canaux du cluster (ex: ['Pz', 'P3', 'P4']).
        tmin: Début de la fenêtre (secondes).
        tmax: Fin de la fenêtre (secondes).
        mode: 'max' (pic positif) ou 'min' (pic négatif).

    Returns:
        Un tuple de deux arrays 1D (chacun de shape [n_essais,]):
        (peak_latencies_sec, peak_amplitudes_uV)
    """
    # 1. Trouver les indices de temps et l'array de temps de la fenêtre
    start, stop = epochs.time_as_index([tmin, tmax])
    times_window = epochs.times[start:stop]
    
    # 2. Obtenir les données 3D (n_essais, n_canaux, n_temps_fenetre)
    data = epochs.get_data(picks=picks)[:, :, start:stop]
    
    # 3. Créer le "canal virtuel" pour chaque essai en moyennant les canaux
    #    Shape -> (n_essais, n_temps_fenetre)
    cluster_signals = data.mean(axis=1)
    
    # 4. Trouver les indices des pics (min ou max) sur l'axe du temps (axis=1)
    if mode == 'max':
        peak_indices = np.argmax(cluster_signals, axis=1)
    elif mode == 'min':
        peak_indices = np.argmin(cluster_signals, axis=1)
    else:
        raise ValueError("Mode doit être 'min' ou 'max'")

    # 5. Extraire les latences (en secondes) en utilisant les indices
    peak_latencies_sec = times_window[peak_indices]
    
    # 6. Extraire les amplitudes (en Volts) en utilisant les indices
    peak_amplitudes_V = np.array([
        cluster_signals[i, idx] for i, idx in enumerate(peak_indices)
    ])
    
    # 7. Convertir en µV et retourner
    return peak_latencies_sec, peak_amplitudes_V * 1e6

In [ ]:
import pandas as pd
import numpy as np

# 1. Initialiser le dictionnaire qui contiendra nos données
features_dict = {}

# Créer une map inversée {code: 'label'} pour traduire les événements
code_to_label = {v: k for k, v in epochs.event_id.items()}

# Ajouter la condition (ex: 'stimulus/go/correct') pour chaque essai
features_dict['condition'] = [code_to_label[code] for code in epochs.events[:, 2]]


# -----------------------------------------------------------------------------
# EXTRACTION DES CARACTÉRISTIQUES DE MOYENNE
# -----------------------------------------------------------------------------
print("  Extracting mean amplitude features...")

for config in ERP_MEAN_WINDOWS:
    name = config['name']
    cluster_name = config['cluster']
    picks = CHANNEL_CLUSTERS[cluster_name]
    tmin = config['tmin']
    tmax = config['tmax']
    
    # Appelle la fonction d'extraction sur les Epochs
    # 'mean_amps' sera un array 1D (shape [n_essais,])
    mean_amps = mean_amplitude_epochs(epochs, picks=picks, tmin=tmin, tmax=tmax)
    
    # Ajoute cet array comme une nouvelle colonne dans notre dictionnaire
    features_dict[name] = mean_amps
    print(f"    -> Feature '{name}' (cluster: {cluster_name}) ajoutée.")


# -----------------------------------------------------------------------------
# EXTRACTION DES CARACTÉRISTIQUES DE PIC
# -----------------------------------------------------------------------------
print("  Extracting peak features...")

for config in ERP_PEAK_WINDOWS:
    name = config['name']
    cluster_name = config['cluster']
    picks = CHANNEL_CLUSTERS[cluster_name]
    tmin = config['tmin']
    tmax = config['tmax']
    mode = config['mode']
    
    # Appelle la fonction d'extraction de pic sur les Epochs
    # 'latencies' et 'amplitudes' sont des arrays 1D
    latencies, amplitudes = peak_latency_epochs(
        epochs, picks=picks, tmin=tmin, tmax=tmax, mode=mode
    )
    
    # Ajoute *deux* nouvelles colonnes pour chaque configuration de pic
    features_dict[name + '_latency_sec'] = latencies
    features_dict[name + '_amplitude_uV'] = amplitudes
    print(f"    -> Features '{name}_latency' et '{name}_amplitude' (cluster: {cluster_name}) ajoutées.")


# -----------------------------------------------------------------------------
# CRÉATION DU DATAFRAME
# -----------------------------------------------------------------------------

# Convertit le dictionnaire de listes/arrays en un DataFrame pandas
df_features = pd.DataFrame(features_dict)

print("\n Extraction des caractéristiques terminée !")
print("\n--- Aperçu du DataFrame (df_features) ---")
print(df_features.head())

print("\n--- Informations sur le DataFrame ---")
df_features.info()


# -----------------------------------------------------------------------------
# Sauvegarde du DataFrame au format CSV
# -----------------------------------------------------------------------------
output_csv = deriv_analysis / f'sub-{subject}_session-{session}_run-{run}_erp_features.csv'
df_features.to_csv(output_csv, index=False)
print(f"\nDataFrame sauvegardé sous : {output_csv}")

### Bloc Type 3 — Explorer d'autres options (similaire au bloc Type 3 du 3.)

Pistes d'exploration :

* **Jouer avec les configurations :** Modifier les dictionnaires `CHANNEL_CLUSTERS`, `ERP_MEAN_WINDOWS`, et `ERP_PEAK_WINDOWS`.
* **Analyse par capteur :** Essayer d'analyser chaque capteur individuellement (sans utiliser de clusters).
* **Ajouter des composantes :** Identifier d'autres ERPs (ex: N1, P2) identifiés dans la littérature et les ajouter aux configurations.

#### Note: Inclure d'autre attributs, peut être une piste d'analyse additionelle comme cela permet de comparer les attribute de bases ainsi que d'autres.

## 4. Extension multi-participants
## 5. Factoriser le pipeline pour tous les sujets

Objectifs : créer des fonctions modulaires qui calcule les attributs automatiquements pour tous les sujets. 

In [ ]:
def process_subject_features(subject: str, session: str, run: str, configs: dict, t_epoch: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    """    
    Exécute le pipeline complet (parsing, epoching, extraction) 
    sur tous les sujets.
    Étapes :
        1. Chargement des données prétraitées.
        2. Parsing des événements.
        3. Création des époques (Epochs).
        4. Extraction des caractéristiques (features).
    
    Args:
        subject (str): ID du sujet (ex: '01').
        session (str): ID de la session (ex: '001').
        run (str): ID du run (ex: '01').
        configs (dict): Dictionnaire contenant les configurations 
                        (CHANNEL_CLUSTERS, ERP_MEAN_WINDOWS, ERP_PEAK_WINDOWS).
        t_epoch (dict): Dictionnaire des temps d'epoching (tmin, tmax, baseline).

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: 
            - df_features: DataFrame des features ERP pour le sujet.
            - df_behavior: DataFrame des données comportementales.
    """
    
    print(f"\n--- Traitement Sujet : {subject} ---")
    
    # --- 0. Décompression des configurations ---
    CHANNEL_CLUSTERS = configs['CHANNEL_CLUSTERS']
    ERP_MEAN_WINDOWS = configs['ERP_MEAN_WINDOWS']
    ERP_PEAK_WINDOWS = configs['ERP_PEAK_WINDOWS']

    # --- 1. Chargement des données ---
    raw = load_processed_raw(subject, session=session, run=run)

    # --- 2. Parsing des événements (Logique "Look-Ahead") ---
    print("  ... 1/5 Parsing des événements...")
    events, event_id = mne.events_from_annotations(raw)
    
    new_event_codes = {
        'stimulus/go/correct': 101, 'stimulus/go/incorrect': 102,
        'stimulus/nogo/correct': 103, 'stimulus/nogo/incorrect': 104
    }
    
    go_onset_code = event_id.get(np.str_('go_onset'))
    nogo_onset_code = event_id.get(np.str_('nogo_onset'))
    go_correct_code = event_id.get(np.str_('go_correct'))
    nogo_correct_code = event_id.get(np.str_('nogo_correct'))
    incorrect_code = event_id.get(np.str_('incorrect'))

    stimulus_codes = [go_onset_code, nogo_onset_code]
    response_codes = [go_correct_code, nogo_correct_code, incorrect_code]

    new_events_list = []
    behavioral_data = []
    sfreq = raw.info['sfreq']
    trial_count = 0

    for i in range(len(events)):
        current_code = events[i, 2]
        if current_code not in stimulus_codes:
            continue
            
        trial_count += 1
        t_onset = events[i, 0]
        stim_type = 'go' if current_code == go_onset_code else 'nogo'
        
        found_response = False
        for j in range(i + 1, len(events)):
            next_code = events[j, 2]
            
            if next_code in response_codes:
                found_response = True
                t_response = events[j, 0]
                rt_sec = (t_response - t_onset) / sfreq
                
                if stim_type == 'go':
                    new_code = new_event_codes['stimulus/go/correct'] if next_code == go_correct_code else new_event_codes['stimulus/go/incorrect']
                    response = 'correct' if next_code == go_correct_code else 'incorrect'
                else: # nogo
                    if next_code == nogo_correct_code:
                        new_code = new_event_codes['stimulus/nogo/correct']
                        response = 'correct'
                        rt_sec = np.nan
                    else:
                        new_code = new_event_codes['stimulus/nogo/incorrect']
                        response = 'incorrect'
                break
                
            if next_code in stimulus_codes:
                found_response = False
                break

        if not found_response:
            rt_sec = np.nan
            if stim_type == 'go':
                new_code = new_event_codes['stimulus/go/incorrect'] # Omission
                response = 'incorrect'
            else: # nogo
                new_code = new_event_codes['stimulus/nogo/correct'] # Omission correcte
                response = 'correct'

        new_events_list.append([t_onset, 0, new_code])
        behavioral_data.append({'subject': subject, 'trial': trial_count, 
                                'stimulus': stim_type, 'response': response, 'rt_sec': rt_sec})

    events_modified = np.array(new_events_list)
    selected_event_id = {label: code for label, code in new_event_codes.items() 
                         if code in events_modified[:, 2]}

    # --- 3. Création des Époques ---
    print("  ... 2/5 Création des époques...")
    epochs = mne.Epochs(
        raw,
        events_modified,
        event_id=selected_event_id,
        tmin=t_epoch['tmin'],
        tmax=t_epoch['tmax'],
        baseline=t_epoch['baseline'],
        picks='eeg',
        preload=True,
        detrend=None,
    )

    # --- 4. Extraction des Caractéristiques (Features) ---
    print("  ... 3/5 Extraction des caractéristiques (features)...")
    features_dict = {}

    # Ajouter les infos de base
    features_dict['subject'] = [subject] * len(epochs)
    code_to_label = {v: k for k, v in epochs.event_id.items()}
    features_dict['condition'] = [code_to_label[code] for code in epochs.events[:, 2]]

    # Extractions (Moyennes)
    for config in ERP_MEAN_WINDOWS:
        name = config['name']
        picks = CHANNEL_CLUSTERS[config['cluster']]
        mean_amps = mean_amplitude_epochs(epochs, picks=picks, tmin=config['tmin'], tmax=config['tmax'])
        features_dict[name] = mean_amps

    # Extractions (Pics)
    for config in ERP_PEAK_WINDOWS:
        name = config['name']
        picks = CHANNEL_CLUSTERS[config['cluster']]
        latencies, amplitudes = peak_latency_epochs(
            epochs, picks=picks, tmin=config['tmin'], tmax=config['tmax'], mode=config['mode']
        )
        features_dict[name + '_latency_sec'] = latencies
        features_dict[name + '_amplitude_uV'] = amplitudes

    # --- 5. Création des DataFrames ---
    print("  ... 4/5 Création des DataFrames...")
    df_features = pd.DataFrame(features_dict)
    df_behavior = pd.DataFrame(behavioral_data)
    
    print(f"  ... 5/5 Sujet {subject} terminé. {len(df_features)} essais extraits.")
    
    return df_features, df_behavior

# --- Définir les Constantes de l'Analyse ---
# Définir la liste des sujets, la session et le run
subjects_list = ['01', '02', '03', '04', '05', '06', '07', '08', '10', '12', '13', '14']
session = '001'
run = '01'

# --- Définir les Configurations de Features ---
CHANNEL_CLUSTERS = {
    "midline_fc": ["Fz", "Cz"],
    "parietal": ["P3", "P4"],
    "motor_left": ["C3"],
    "motor_right": ["C4"],
}

ERP_MEAN_WINDOWS = [
    {"name": "N2_mean_200_350", "cluster": "midline_fc", "tmin": 0.200, "tmax": 0.350},
    {"name": "P3_mean_300_600_midline_fc", "cluster": "midline_fc", "tmin": 0.300, "tmax": 0.600},
    {"name": "P3_mean_300_600_parietal", "cluster": "parietal", "tmin": 0.300, "tmax": 0.600},
]

ERP_PEAK_WINDOWS = [
    {"name": "N2_peak_200_350", "cluster": "midline_fc", "tmin": 0.200, "tmax": 0.350, "mode": "min"},
    {"name": "P3_peak_300_600", "cluster": "parietal", "tmin": 0.300, "tmax": 0.600, "mode": "max"},
]

# Regrouper les configurations
configs = {
    "CHANNEL_CLUSTERS": CHANNEL_CLUSTERS,
    "ERP_MEAN_WINDOWS": ERP_MEAN_WINDOWS,
    "ERP_PEAK_WINDOWS": ERP_PEAK_WINDOWS
}

# Définir les temps d'epoching
t_epoch = {
    "tmin": -0.2,
    "tmax": 0.8,
    "baseline": (-0.2, 0.0)
}

# --- Lancer la Boucle ---
all_features_dfs = []
all_behavior_dfs = []

print(f"Lancement du pipeline pour {len(subjects_list)} sujets...")

for subject in subjects_list:
    df_feat, df_behav = process_subject_features(
        subject=subject, 
        session=session, 
        run=run,
        configs=configs,
        t_epoch=t_epoch
    )
    
    all_features_dfs.append(df_feat)
    all_behavior_dfs.append(df_behav)
            

if all_features_dfs and all_behavior_dfs:
    print("\n--- Pipeline terminé. Combinaison des résultats... ---")
    
    # Combiner tous les DataFrames individuels en un seul
    final_features_df = pd.concat(all_features_dfs, ignore_index=True)
    final_behavior_df = pd.concat(all_behavior_dfs, ignore_index=True)

    # Sauvegarder les fichiers CSV finaux
    output_features_csv = deriv_analysis / "all_subjects_erp_features.csv"
    output_behavior_csv = deriv_analysis / "all_subjects_behavioral_data.csv"
    
    final_features_df.to_csv(output_features_csv, index=False)
    final_behavior_df.to_csv(output_behavior_csv, index=False)
    
    print(f"DataFrame de features (attributs) final ({final_features_df.shape}):")
    print(final_features_df.head())
    
    print(f"\nDataFrame comportemental final ({final_behavior_df.shape}):")
    print(final_behavior_df.head())
    
    print(f"\nRésultats sauvegardés dans : {deriv_analysis}")

else:
    print("\n--- ⚠️ Aucun sujet n'a été traité avec succès. ---")

## 5. Synthèse et prochaines étapes

Vous avez maintenant :
1.  Implémenté une **logique d'analyse d'événements** complexe pour transformer les annotations brutes en époques *stimulus-locked* significatives (ex: `stimulus/go/correct`).
2.  Extrait et structuré les **données comportementales** (temps de réaction, justesse) pour chaque essai.
3.  Défini des **fonctions d'extraction de caractéristiques (features)** robustes (`mean_amplitude_epochs`, `peak_latency_epochs`) basées sur des clusters de canaux pré-définis.
4.  Construit et automatisé un pipeline qui génère un **DataFrame  unique**, contenant les features et les étiquettes (`y`) pour *tous les sujets*.

Prochaine étape :. Ce DataFrame est maintenant parfaitement préparé pour être utilisé comme matrice `X` (vos attributs) et `y` (vos cibles) afin d'entraîner et d'évaluer des modèles de classification pour prédire la condition (Go vs NoGo) à partir de l'activité cérébrale.